# Qwen3-1.7B DPO — POC Triage Médical CHSA

Alignement par préférences (DPO) du modèle **Qwen3-1.7B SFT** pour améliorer la classification triage P1/P2/P3.

**Pipeline :** Qwen3-1.7B-Base → SFT (LoRA) ✅ → **DPO** → Endpoint vLLM  
**Repo :** [XavierCoulon/OC_P14_Finetunez_votre_propre_LLM](https://github.com/XavierCoulon/OC_P14_Finetunez_votre_propre_LLM)  
**Modèle SFT :** [XavierCoulon/qwen3-1.7b-chsa-sft-lora](https://huggingface.co/XavierCoulon/qwen3-1.7b-chsa-sft-lora)  
**Modèle DPO publié :** [XavierCoulon/qwen3-1.7b-chsa-dpo](https://huggingface.co/XavierCoulon/qwen3-1.7b-chsa-dpo)

> Exécuter sur Kaggle ou Google Colab avec GPU (T4 minimum).  
> Prérequis Secrets : `HF_TOKEN`, `WANDB_API_KEY`.

In [ ]:
%%capture
import os, re

env_keys = "".join(os.environ.keys())
ON_COLAB  = "COLAB_" in env_keys
ON_KAGGLE = "KAGGLE_" in env_keys

if ON_COLAB or ON_KAGGLE:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
else:
    !pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install wandb

In [ ]:
def get_secret(name):
    """Lit un secret depuis Kaggle, Colab ou variable d'environnement."""
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    if ON_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    return os.environ.get(name, "")

import wandb
wandb.login(key=get_secret('WANDB_API_KEY'))
wandb.init(project="chsa-dpo-qwen3", name="run-v1")

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None
load_in_4bit = True

# Charge le modèle SFT (adapters LoRA)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "XavierCoulon/qwen3-1.7b-chsa-sft-lora",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

### Data Prep DPO

Dataset de préférences `XavierCoulon/oc-p14-dataset` (config `dpo`) — 1 600 paires `chosen`/`rejected` issues de UltraMedical-Preference.

Format attendu par DPOTrainer : `prompt` / `chosen` / `rejected` en format ChatML Qwen3.

In [ ]:
import random
random.seed(42)

from datasets import load_dataset

dataset = load_dataset("XavierCoulon/oc-p14-dataset", "dpo", split="train")
print(f"Dataset DPO train : {len(dataset)} paires")

val_raw = load_dataset("XavierCoulon/oc-p14-dataset", "dpo", split="val")
print(f"Dataset DPO val   : {len(val_raw)} paires")
print(dataset.column_names)


In [ ]:
import json, random
random.seed(42)

def format_dpo(examples):
    prompts, chosens, rejecteds = [], [], []
    for prompt, chosen, rejected in zip(
        examples["prompt"], examples["chosen"], examples["rejected"]
    ):
        think_tag = "/think" if random.random() < 0.75 else "/no_think"
        p = (
            f"<|im_start|>user\n{think_tag}\n{prompt}<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
        # chosen/rejected sont des dicts {"role": "assistant", "content": "..."}
        if isinstance(chosen, str):
            chosen = json.loads(chosen)
        if isinstance(rejected, str):
            rejected = json.loads(rejected)
        prompts.append(p)
        chosens.append(chosen["content"] + tokenizer.eos_token)
        rejecteds.append(rejected["content"] + tokenizer.eos_token)
    return {"prompt": prompts, "chosen": chosens, "rejected": rejecteds}

dataset = dataset.map(format_dpo, batched=True, remove_columns=dataset.column_names)
val_dataset = val_raw.map(format_dpo, batched=True, remove_columns=val_raw.column_names)
print(f"Train formaté : {len(dataset)} paires")
print(f"Val formaté   : {len(val_dataset)} paires")
print("\nExemple prompt :")
print(dataset[0]["prompt"][:200])
print("\nChosen :")
print(dataset[0]["chosen"][:200])


### Entraînement DPO

DPOTrainer avec LoRA — 1 epoch sur les 1 600 paires. `beta=0.1` contrôle l'écart par rapport au modèle de référence (SFT).

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")

In [ ]:
from trl import DPOConfig, DPOTrainer

trainer = DPOTrainer(
    model = model,
    ref_model = None,   # None = utilise le modèle de base gelé (recommandé avec LoRA)
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = val_dataset,
    args = DPOConfig(
        beta = 0.1,
        max_length = 1024,
        max_prompt_length = 512,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        num_train_epochs = 1,
        learning_rate = 5e-5,
        lr_scheduler_type = "cosine",
        warmup_ratio = 0.1,
        optim = "adamw_8bit",
        seed = 42,
        output_dir = "outputs_dpo",
        report_to = "wandb",
        save_strategy = "steps",
        save_steps = 100,
        save_total_limit = 2,
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = 100,
    ),
)


In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"{trainer_stats.metrics['train_runtime']/60:.2f} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")

### Sauvegarde et push vers HuggingFace Hub

In [ ]:
HF_TOKEN  = get_secret('HF_TOKEN')
HF_REPO   = "XavierCoulon/qwen3-1.7b-chsa-dpo"

model.save_pretrained("qwen_dpo_lora")
tokenizer.save_pretrained("qwen_dpo_lora")

model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f"Modèle DPO publié : https://huggingface.co/{HF_REPO}")

### Fusion pour déploiement vLLM

vLLM nécessite un modèle complet (pas des adapters LoRA). Cette cellule fusionne les poids et publie le modèle merged.

In [ ]:
HF_REPO_MERGED = "XavierCoulon/qwen3-1.7b-chsa-dpo-merged"

model.push_to_hub_merged(
    HF_REPO_MERGED,
    tokenizer,
    save_method = "merged_16bit",
    token = HF_TOKEN,
)
print(f"Modèle merged publié : https://huggingface.co/{HF_REPO_MERGED}")
print("→ Utiliser ce modèle dans docker-compose.yml (MODEL_NAME)")

In [ ]:
wandb.finish()